# Tests

## Test the agent against other agents

In [27]:
import numpy as np
import torch

from connect4 import Connect4Env
from utils import preprocess_board

from AlphaBetaBot import AlphaBetaBot

from policy import Policy


def test_sigma_agent(env_class, policy = None, policy_2: None|Policy|AlphaBetaBot = None, games=100, visual=False): # (made with gemini)
    env = env_class()
    ai_wins = 0
    
    print(f"Running {games} games against other Bot...")
    
    for game in range(games):
        obs, _ = env.reset()
        # AI plays as Player 1 (Blue)
        done = False
        
        while not done:
            if env.current_player == 1: # AI Turn
                # AI Logic
                if policy is not None:
                    ai_input = torch.from_numpy(preprocess_board(obs, 1)).float().unsqueeze(0)
                    with torch.no_grad():
                        valid_actions_mask = torch.tensor([obs[0][col] == 0 for col in range(7)], dtype=torch.bool)
                        # The mask is boolean, so we apply it to the logits
                        # We want to set the logits of invalid actions to a very small number
                        # so that they are not chosen by the categorical distribution.
                        logits = policy.forward(ai_input)[0]
                        logits[~valid_actions_mask] = -float('inf')
                        # mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)])
                        action = torch.argmax(logits).item()
                else:
                    print("please give policy to check")
                    return
            else: # Random Turn
                if policy_2 is not None and isinstance(policy_2, Policy):
                    ai_input = torch.from_numpy(preprocess_board(obs, -1)).float().unsqueeze(0)
                    with torch.no_grad():
                        logits = policy_2.forward(ai_input)[0]
                        mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)])
                        action = torch.argmax(logits + mask).item()
                elif policy_2 is not None and isinstance(policy_2, AlphaBetaBot):
                    action = policy_2.get_action(obs, -1)
                    
                else:
                    # Random Logic
                    legal = [c for c in range(env.cols) if env.board[0][c] == 0]
                    action = np.random.choice(legal)
                
            obs, _, done, _, info = env.step(action)
            if visual:
                env.render()
            
        if info['winner'] == 1:
            if visual:
                print("AI Wins!")
            ai_wins += 1
            
    print(f"Sigma AI Win Rate vs Bot: {ai_wins}/{games} ({ai_wins/games*100}%)")






    

    

In [36]:
env = Connect4Env()
from policy import Policy
from state_value import StateValue
ppo_policy = Policy(env.action_space.n, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, conv_layers_channels=[128, 64, 32], fc_layer_sizes=[512, 512, 256])
ppo_policy.load_from_file("This bot is super good large CNN.pth")
ppo_policy.eval()
policy_2 = AlphaBetaBot(depth=9)
test_sigma_agent(Connect4Env, policy=ppo_policy, policy_2=policy_2, games=1, visual=True)

Loading from: This bot is super good large CNN.pth to cuda
SUCCESS: Model loaded.
Running 1 games against other Bot...
              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
              
              

              
              
              
              
  

## Check if the board is being prepared correctly

In [6]:
def run_diagnostic_prepare_board():
    print("\n--- INITIATING PREPROCESSING DIAGNOSTIC ---")
    
    # Mock Board: 6 rows, 7 cols
    # Row 5 (Bottom): [P1, P2, Empty, ...]
    board = np.zeros((6, 7), dtype=np.int8)
    board[5][0] = 1   # Player 1 is at Bottom-Left
    board[5][1] = -1  # Player 2 is next to him
    
    print("State of the Board:")
    print("Row 5: [ P1(1), P2(-1), 0, 0 ... ]")

    # --- TEST CASE A: Player 1's Perspective ---
    print("\n[TEST A] Player 1's Turn (current_player = 1)")
    p1_obs = preprocess_board(board, current_player=1)
    
    # Check Shape
    if p1_obs.shape == (2, 6, 7):
        print("  [PASS] Shape is correct (2, 6, 7).")
    else:
        print(f"  [FAIL] Shape is {p1_obs.shape}.")

    # Check Channel 0 (Should see P1 at [5][0])
    if p1_obs[0][5][0] == 1.0 and p1_obs[0][5][1] == 0.0:
        print("  [PASS] Channel 0 (Me) correctly identifies P1's piece.")
    else:
        print(f"  [FAIL] Channel 0 failed. \n{p1_obs[0]}")

    # Check Channel 1 (Should see P2 at [5][1])
    if p1_obs[1][5][1] == 1.0 and p1_obs[1][5][0] == 0.0:
        print("  [PASS] Channel 1 (Enemy) correctly identifies P2's piece.")
    else:
        print(f"  [FAIL] Channel 1 failed. \n{p1_obs[1]}")

    
    # --- TEST CASE B: Player 2's Perspective (The "Flip" Test) ---
    print("\n[TEST B] Player 2's Turn (current_player = -1)")
    # To Player 2: 
    #   P2 (at 5,1) is "Me" -> Should be in Channel 0
    #   P1 (at 5,0) is "Enemy" -> Should be in Channel 1
    p2_obs = preprocess_board(board, current_player=-1)

    # Check Channel 0 (Me) -> Should contain the piece at [5][1]
    if p2_obs[0][5][1] == 1.0 and p2_obs[0][5][0] == 0.0:
        print("  [PASS] Channel 0 (Me) correctly grabbed P2's piece (The Flip worked).")
    else:
        print(f"  [FAIL] Channel 0 failed to flip perspective. \n{p2_obs[0]}")

    # Check Channel 1 (Enemy) -> Should contain the piece at [5][0]
    if p2_obs[1][5][0] == 1.0:
        print("  [PASS] Channel 1 (Enemy) correctly identified P1 as the enemy.")
    else:
        print(f"  [FAIL] Channel 1 failed to identify enemy. \n{p2_obs[1]}")

    print("\n--- DIAGNOSTIC COMPLETE ---")

In [7]:
run_diagnostic_prepare_board()


--- INITIATING PREPROCESSING DIAGNOSTIC ---
State of the Board:
Row 5: [ P1(1), P2(-1), 0, 0 ... ]

[TEST A] Player 1's Turn (current_player = 1)
  [PASS] Shape is correct (2, 6, 7).
  [PASS] Channel 0 (Me) correctly identifies P1's piece.
  [PASS] Channel 1 (Enemy) correctly identifies P2's piece.

[TEST B] Player 2's Turn (current_player = -1)
  [PASS] Channel 0 (Me) correctly grabbed P2's piece (The Flip worked).
  [PASS] Channel 1 (Enemy) correctly identified P1 as the enemy.

--- DIAGNOSTIC COMPLETE ---


## Check if the trajectories are configured correctly

In [8]:
def test_trajectory_integrity(env_class):
    print("\n--- INITIATING SIGMA 2-CHANNEL DIAGNOSTIC ---")
    env = env_class()
    obs, _ = env.reset()
    
    # Force a game: P1 wins in Col 0
    # Moves: P1(0), P2(1), P1(0), P2(1), P1(0), P2(1), P1(0)
    forced_actions = [0, 1, 0, 1, 0, 1, 0]
    
    recorded_obs = []
    recorded_actions = []
    recorded_rewards = []
    
    print("Simulating Game...")
    
    for i, action in enumerate(forced_actions):
        # 1. CAPTURE STATE (Process it exactly like the Neural Network will)
        # Note: 'obs' here is the raw board from env.reset/step
        processed_obs = preprocess_board(env.board, env.current_player)
        recorded_obs.append(processed_obs)
        
        # 2. CAPTURE ACTION
        recorded_actions.append(action)
        
        # 3. EXECUTE STEP
        next_obs, reward, done, _, info = env.step(action)
        
        # 4. CAPTURE REWARD
        recorded_rewards.append(reward)
        
        # Update raw obs for next loop (though env.board is the source of truth)
        obs = next_obs
        
        if done:
            print(f"Game Ended at step {i}. Winner: {info['winner']}")
            break

    # --- VERIFICATION PHASE ---
    print("\n--- ANALYZING TIMELINE ---")
    
    # CHECK 1: Reward Alignment
    last_reward = recorded_rewards[-1]
    if last_reward == 1:
        print("[PASSED] Reward Alignment: Winning move got +1.0.")
    else:
        print(f"[FAILED] Reward Alignment: Expected 1.0, got {last_reward}.")

    # CHECK 2: State Causality (Step 0 should be empty)
    step_0_obs = recorded_obs[0] # Shape (2, 6, 7)
    if np.all(step_0_obs == 0):
        print("[PASSED] State Causality: Step 0 is totally empty (Correct).")
    else:
        print("[FAILED] State Causality: Step 0 is NOT empty. Future leak detected!")
        # print(step_0_obs)

    # CHECK 3: The "Future" Leak (Step 0 should not see the move at Col 0)
    # Channel 0 (My Pieces) at bottom-left [5][0] should be 0.
    if step_0_obs[0][5][0] == 0:
         print("[PASSED] No Future Leak: Step 0 does not see the move it is about to make.")
    else:
         print("[FAILED] Future Leak Detected! Step 0 sees itself at [5][0].")

    # CHECK 4: Perspective & Channels (The Big One)
    # At Step 1, it is Player 2's turn.
    # Player 1 (Enemy) is at Col 0.
    # Player 2 should see a piece in Channel 1 (Enemy Channel) at Col 0.
    step_1_obs = recorded_obs[1] # Player 2's view
    
    # Check Channel 0 (Me/P2) -> Should be empty (P2 hasn't moved yet)
    is_me_empty = np.all(step_1_obs[0] == 0)
    
    # Check Channel 1 (Enemy/P1) -> Should have piece at [5][0]
    has_enemy = (step_1_obs[1][5][0] == 1)
    
    if is_me_empty and has_enemy:
        print("[PASSED] Perspective Flip: Player 2 correctly sees P1 in the 'Enemy Channel'.")
    else:
        print(f"[FAILED] Perspective Flip: \n  Me Empty? {is_me_empty}\n  Enemy Detected? {has_enemy}")
        print(f"  Channel 0 (Me):\n{step_1_obs[0]}")
        print(f"  Channel 1 (Enemy):\n{step_1_obs[1]}")

    print("--- DIAGNOSTIC COMPLETE ---")

In [9]:
env = Connect4Env()
test_trajectory_integrity(Connect4Env)



--- INITIATING SIGMA 2-CHANNEL DIAGNOSTIC ---
Simulating Game...
Game Ended at step 6. Winner: 1

--- ANALYZING TIMELINE ---
[PASSED] Reward Alignment: Winning move got +1.0.
[PASSED] State Causality: Step 0 is totally empty (Correct).
[PASSED] No Future Leak: Step 0 does not see the move it is about to make.
[PASSED] Perspective Flip: Player 2 correctly sees P1 in the 'Enemy Channel'.
--- DIAGNOSTIC COMPLETE ---


## Test check_win functions (fast vs slow)

In [10]:
import time
import numpy as np
from numba import njit
import random

# --- 1. THE OLD LOGIC (Slow, Global Scan) ---
def check_win_slow(board, piece):
    rows, cols = 6, 7
    # Horizontal
    for c in range(cols - 3):
        for r in range(rows):
            if board[r][c] == piece and board[r][c+1] == piece and board[r][c+2] == piece and board[r][c+3] == piece:
                return True
    # Vertical
    for c in range(cols):
        for r in range(rows - 3):
            if board[r][c] == piece and board[r+1][c] == piece and board[r+2][c] == piece and board[r+3][c] == piece:
                return True
    # Pos Slope
    for c in range(cols - 3):
        for r in range(rows - 3):
            if board[r][c] == piece and board[r+1][c+1] == piece and board[r+2][c+2] == piece and board[r+3][c+3] == piece:
                return True
    # Neg Slope
    for c in range(cols - 3):
        for r in range(3, rows):
            if board[r][c] == piece and board[r-1][c+1] == piece and board[r-2][c+2] == piece and board[r-3][c+3] == piece:
                return True
    return False

@njit
def check_win_fast(board, row, col, piece):
    # Constants for clarity
    ROWS = 6
    COLS = 7

    # 1. Check Horizontal (-)
    count = 0
    # Optimization: We only need to check the row 'row'
    for c in range(COLS):
        if board[row][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0

    # 2. Check Vertical (|)
    count = 0
    # Optimization: We only need to check the col 'col'
    for r in range(ROWS):
        if board[r][col] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
    
    # 3. Check Main Diagonal (\) 
    # Direction: Top-Left to Bottom-Right
    # We find the Top-Left-most start point
    count = 0
    offset = min(row, col)
    r = row - offset
    c = col - offset
    
    while r < ROWS and c < COLS:
        if board[r][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
        r += 1
        c += 1

    # 4. Check Anti-Diagonal (/) 
    # Direction: Top-Right to Bottom-Left
    # We find the Top-Right-most start point
    count = 0
    
    # FIX IS HERE: Distance to right edge is (COLS - 1) - col
    offset = min(row, (COLS - 1) - col) 
    
    r = row - offset
    c = col + offset
    
    while r < ROWS and c >= 0:
        if board[r][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
        r += 1
        c -= 1
        
    return False

# --- 3. THE SIGMA BENCHMARK ---
def run_comparison(num_checks=100000):
    print(f"Running validation on {num_checks} simulated moves...")
    
    board = np.zeros((6, 7), dtype=np.int8)
    
    # Time Trackers
    start_slow = 0
    total_slow = 0
    
    start_fast = 0
    total_fast = 0
    
    # Compile the Numba function once before timing (warmup)
    check_win_fast(board, 0, 0, 1)

    errors = 0
    
    for _ in range(num_checks):
        # 1. Generate a random move
        col = random.randint(0, 6)
        
        # Find row (simple manual find)
        row = -1
        for r in range(5, -1, -1):
            if board[r][col] == 0:
                row = r
                break
        
        # If column full, reset board
        if row == -1:
            board.fill(0)
            continue
            
        piece = 1 if random.random() > 0.5 else -1
        board[row][col] = piece
        
        # 2. Benchmark SLOW
        t0 = time.perf_counter()
        res_slow = check_win_slow(board, piece)
        t1 = time.perf_counter()
        total_slow += (t1 - t0)
        
        # 3. Benchmark FAST
        t0 = time.perf_counter()
        res_fast = check_win_fast(board, row, col, piece)
        t1 = time.perf_counter()
        total_fast += (t1 - t0)
        
        # 4. Verification
        if res_slow != res_fast:
            print(f"\n[CRITICAL ERROR] Mismatch found!")
            print(board)
            print(f"Move: Row {row}, Col {col}, Piece {piece}")
            print(f"Slow says: {res_slow}, Fast says: {res_fast}")
            errors += 1
            break
            
        # Reset board if someone won, to keep simulating games
        if res_slow:
            board.fill(0)

    if errors == 0:
        print("\n✅ SUCCESS: logic matches perfectly across all simulations.")
        print("-" * 40)
        print(f"Slow Logic Total Time: {total_slow:.4f}s")
        print(f"Fast Logic Total Time: {total_fast:.4f}s")
        print(f"Speedup Factor: {total_slow / total_fast:.2f}x faster")
        print("-" * 40)



In [11]:
if __name__ == "__main__":
    run_comparison()

Running validation on 100000 simulated moves...

✅ SUCCESS: logic matches perfectly across all simulations.
----------------------------------------
Slow Logic Total Time: 2.3051s
Fast Logic Total Time: 0.0608s
Speedup Factor: 37.92x faster
----------------------------------------


## Test how fast the gpu is against the cpu

In [12]:
from policy import Policy
import torch

def run_inference_benchmark(device_name, batch_size, num_runs=1000):
    if device_name == "cuda" and not torch.cuda.is_available():
        print("❌ CUDA not available. Skipping GPU test.")
        return 0.0

    device = torch.device(device_name)
    model = Policy(7, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, conv_layers_channels=[32, 64, 32], fc_layer_sizes=[256, 256, 256]).to(device)
    model.eval() # Inference mode

    # Create dummy input
    input_data = torch.randn(batch_size, 2, 6, 7).to(device)

    # Warmup (Wake up the GPU)
    for _ in range(10):
        _ = model(input_data)
    
    # Synchronize before starting timer (Crucial for GPU)
    if device_name == "cuda":
        torch.cuda.synchronize()
    
    start_time = time.perf_counter()
    
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(input_data)
            
    # Synchronize after finishing (Crucial for GPU)
    if device_name == "cuda":
        torch.cuda.synchronize()

    end_time = time.perf_counter()
    
    avg_time = (end_time - start_time) / num_runs
    print(f"[{device_name.upper()}] Batch Size {batch_size}: {avg_time*1000:.4f} ms per pass")
    return avg_time


In [13]:
def run_training_benchmark(device_name, batch_size=1024, num_runs=50):
    """
    Measures how fast the device can perform a full PPO update:
    Forward Pass + Loss Calc + Backward Pass + Optimizer Step
    """
    if device_name == "cuda" and not torch.cuda.is_available():
        print("❌ CUDA not available. Skipping GPU test.")
        return 0.0

    device = torch.device(device_name)
    print(f"Preparing Training Benchmark on {device_name.upper()}...")

    # Initialize Policy
    model = Policy(7, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, 
                   conv_layers_channels=[32, 64, 32], fc_layer_sizes=[256, 256, 256]).to(device)
    model.train() # Training mode

    # Create Dummy Batch Data (Simulating a replay buffer)
    # Obs: (Batch, 2, 6, 7)
    obs = torch.randn(batch_size, 2, 6, 7).to(device)
    # Actions: (Batch,) - Random integers 0-6
    actions = torch.randint(0, 7, (batch_size,)).float().to(device)
    # Old Log Probs: (Batch,)
    old_probs = torch.randn(batch_size).to(device)
    # Rewards (Returns): (Batch,)
    rewards = torch.randn(batch_size).to(device)
    # Advantages: (Batch,)
    advantages = torch.randn(batch_size).to(device)

    # Warmup
    for _ in range(5):
        model.optimizer_step(obs, actions, old_probs, rewards, advantages)

    if device_name == "cuda": torch.cuda.synchronize()
    
    start_time = time.perf_counter()
    
    # Run the Loop
    for _ in range(num_runs):
        model.optimizer_step(obs, actions, old_probs, rewards, advantages)

    if device_name == "cuda": torch.cuda.synchronize()

    end_time = time.perf_counter()
    avg_time = (end_time - start_time) / num_runs
    print(f"[{device_name.upper()}] Training  (Batch {batch_size}): {avg_time*1000:.4f} ms per step")
    return avg_time

In [14]:
print(f"--- BENCHMARKING THE SIGMA AGENT ---")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 1. Inference Tests
print("\n--- Phase 1: Inference ---")
run_inference_benchmark("cpu", 1, num_runs=1000)
run_inference_benchmark("cuda", 1, num_runs=1000)
print("...")
run_inference_benchmark("cpu", 1024, num_runs=100)
run_inference_benchmark("cuda", 1024, num_runs=100)



--- BENCHMARKING THE SIGMA AGENT ---
GPU: NVIDIA GeForce RTX 4080 SUPER

--- Phase 1: Inference ---
[CPU] Batch Size 1: 0.4216 ms per pass
[CUDA] Batch Size 1: 0.9923 ms per pass
...
[CPU] Batch Size 1024: 14.5241 ms per pass
[CUDA] Batch Size 1024: 1.7655 ms per pass


0.0017654529999708757

In [15]:
# 2. Training Tests
print("\n--- Phase 2: Training (Backpropagation) ---")
print("Simulating PPO Update with Batch Size 1024")

cpu_train_time = run_training_benchmark("cpu", 256, num_runs=20)
gpu_train_time = run_training_benchmark("cuda", 256, num_runs=100)

if gpu_train_time > 0:
    speedup = cpu_train_time / gpu_train_time
    print(f"\n🚀 FINAL VERDICT: GPU is {speedup:.2f}x FASTER for Training.")


--- Phase 2: Training (Backpropagation) ---
Simulating PPO Update with Batch Size 1024
Preparing Training Benchmark on CPU...
[CPU] Training  (Batch 256): 30.7469 ms per step
Preparing Training Benchmark on CUDA...
[CUDA] Training  (Batch 256): 10.9581 ms per step

🚀 FINAL VERDICT: GPU is 2.81x FASTER for Training.


## How fast is each game

In [17]:
from policy import Policy
import torch
import time
from connect4 import Connect4Env
from utils import preprocess_board

def benchmark_game_speed(env, policy, device, n_games=1000):
    print(f"\n--- 🏎️  STARTING SPEED BENCHMARK ({n_games} Games) ---")
    print(f"Device: {device}")
    
    # Switch to eval mode (disable dropout, etc. for pure speed)
    policy.eval()
    
    start_time = time.perf_counter()
    total_steps = 0
    
    for i in range(n_games):
        obs, _ = env.reset()
        terminated = False
        
        while not terminated:
            # 1. Preprocess
            current_obs = preprocess_board(obs, env.current_player)
            
            # 2. Inference (GPU)
            with torch.no_grad():
                tensor_input = torch.from_numpy(current_obs).unsqueeze(0).float().to(device)
                
                # Fast Policy Inference
                logits = policy(tensor_input)[0]
                
                # Fast Masking
                mask = torch.tensor([0.0 if env.board[0][c] == 0 else -1e9 for c in range(env.cols)], device=device)
                
                # Greedy action for benchmarking (faster than sampling)
                # or sample if you want realistic simulation
                action = torch.argmax(logits + mask).item() 
            
            # 3. Environment Step (Numba Optimized)
            obs, _, terminated, _, _ = env.step(action)
            total_steps += 1
            
    end_time = time.perf_counter()
    duration = end_time - start_time
    
    # --- REPORT CARD ---
    avg_time_per_game = (duration / n_games) * 1000 # in ms
    games_per_sec = n_games / duration
    steps_per_sec = total_steps / duration
    
    print(f"--- 🏁 BENCHMARK COMPLETE ---")
    print(f"Total Time:       {duration:.4f} seconds")
    print(f"Total Steps:      {total_steps}")
    print(f"Avg Game Time:    {avg_time_per_game:.2f} ms")
    print("-" * 30)
    print(f"⚡ Games Per Second: {games_per_sec:.2f}")
    print(f"⚡ Steps Per Second: {steps_per_sec:.2f}")
    
    # The Sigma Verdict
    if games_per_sec > 100:
        print("Verdict: HYPERSONIC. You are ready for massive scale.")
    elif games_per_sec > 50:
        print("Verdict: SUPERSONIC. Respectable speed.")
    else:
        print("Verdict: SUBSONIC. We might need Vectorized Environments.")

In [18]:
ppo_policy = Policy(7, input_channels=2, board_height=6, board_width=7, ent_coef=0.03, conv_layers_channels=[32, 64, 32], fc_layer_sizes=[256, 256, 256]).to("cuda")
env = Connect4Env()
benchmark_game_speed(env, ppo_policy, "cuda", n_games=500)
benchmark_game_speed(env, ppo_policy.cpu(), "cpu", n_games=500)


--- 🏎️  STARTING SPEED BENCHMARK (500 Games) ---
Device: cuda
--- 🏁 BENCHMARK COMPLETE ---
Total Time:       15.2639 seconds
Total Steps:      9500
Avg Game Time:    30.53 ms
------------------------------
⚡ Games Per Second: 32.76
⚡ Steps Per Second: 622.38
Verdict: SUBSONIC. We might need Vectorized Environments.

--- 🏎️  STARTING SPEED BENCHMARK (500 Games) ---
Device: cpu
--- 🏁 BENCHMARK COMPLETE ---
Total Time:       5.1680 seconds
Total Steps:      9500
Avg Game Time:    10.34 ms
------------------------------
⚡ Games Per Second: 96.75
⚡ Steps Per Second: 1838.24
Verdict: SUPERSONIC. Respectable speed.


# Benchmark Bots against Solved bot

In [25]:
import importlib
import extract_architectures
importlib.reload(extract_architectures)
from extract_architectures import create_architectures_json
create_architectures_json()

Extracting architecture from Saves/bots/beat_alpha_beta_bot.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_2000.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_20000.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_20500.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_21000.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_21500.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_22000.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_22500.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_23000.pth
Extracting architecture from Saves/bots\connect4_parallel_iter_2500.pth
Extracting architecture from Saves/unused\Best RNN model.pth
Extracting architecture from Saves/unused\best small CNN model.pth
Extracting architecture from Saves/unused\current_model.pth
Error inspecting Saves/unused\current_model.pth: PytorchStreamReader failed locating file dat

In [ ]:
import importlib
import extract_architectures
importlib.reload(extract_architectures)
from extract_architectures import create_architectures_json
import numpy as np
import torch
import random
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import os
import json
from collections import OrderedDict
from torch import nn
from torch.distributions import Categorical
import glob
from typing import List

from extract_architectures import create_architectures_json

# ---connect4.py from the past---
import gymnasium as gym
from gymnasium import spaces
from typing import Tuple, Optional, Dict, Any
from numba import njit


@njit
def check_win_fast(board, row, col, piece):
    ROWS, COLS = 6, 7
    # Horizontal
    count = 0
    for c in range(COLS):
        if board[row][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
    # Vertical
    count = 0
    for r in range(ROWS):
        if board[r][col] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
    # Main Diagonal
    count = 0
    offset = min(row, col)
    r, c = row - offset, col - offset
    while r < ROWS and c < COLS:
        if board[r][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
        r, c = r + 1, c + 1
    # Anti-Diagonal
    count = 0
    offset = min(row, (COLS - 1) - col)
    r, c = row - offset, col + offset
    while r < ROWS and c >= 0:
        if board[r][c] == piece:
            count += 1
            if count == 4: return True
        else:
            count = 0
        r, c = r + 1, c - 1
    return False

@njit
def get_next_open_row_fast(board, col):
    for r in range(5, -1, -1):
        if board[r][col] == 0:
            return r
    return -1

class Connect4Env(gym.Env):
    def __init__(self) -> None:
        super(Connect4Env, self).__init__()
        self.rows: int = 6
        self.cols: int = 7
        self.action_space = spaces.Discrete(self.cols)
        self.observation_space = spaces.Box(low=0, high=2, shape=(6, 7), dtype=np.int8)
        self.board: np.ndarray = np.zeros((self.rows, self.cols), dtype=np.int8)
        self.current_player: int = 1
        self.done: bool = False

    def reset(self, *, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
        super().reset(seed=seed)
        self.board.fill(0)
        self.current_player = 1
        self.done = False
        return self.board, {}

    def step(self, action):
        if self.board[0][action] != 0:
            return self.board.copy(), -0.1, True, False, {"error": "Invalid", "winner": 0}
        row = get_next_open_row_fast(self.board, action)
        self.board[row][action] = self.current_player
        if check_win_fast(self.board, row, action, self.current_player):
            self.done = True
            return self.board.copy(), 1.0, True, False, {"winner": self.current_player}
        if np.all(self.board[0] != 0):
            self.done = True
            return self.board.copy(), 0, True, False, {"winner": 0}
        self.current_player *= -1
        return self.board.copy(), 0, False, False, {"winner": 0}

# --- Preprocessing functions ---

def preprocess_board_cnn(board, current_player):
    relative_board = board * current_player
    return np.stack([(relative_board == 1).astype(np.float32), (relative_board == -1).astype(np.float32)])

def preprocess_board_rnn(board, current_player):
    return (board.flatten() * current_player).astype(np.float32)

# --- Bot Architectures ---
from state_value import StateValue
from SolvedBot import SolvedBot
from AlphaBetaBot import AlphaBetaBot

class PolicyCNN(nn.Module):
    def __init__(self, actions_size: int, input_channels: int = 1, conv_layers_channels: List[int] = [32, 64], fc_layer_sizes: List[int] = [256], **kwargs):
        super().__init__()
        self.input_channels = input_channels
        self.board_height, self.board_width = 6, 7
        
        conv_net_layers = []
        in_channels = self.input_channels
        for out_channels in conv_layers_channels:
            conv_net_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            conv_net_layers.append(nn.ReLU())
            in_channels = out_channels
        conv_net_layers.append(nn.Flatten())
        self.conv_net = nn.Sequential(*conv_net_layers)

        with torch.no_grad():
            dummy = torch.zeros(1, self.input_channels, self.board_height, self.board_width)
            conv_out_size = self.conv_net(dummy).shape[1]
        
        fc_net_layers = []
        in_features = conv_out_size
        for out_features in fc_layer_sizes:
            fc_net_layers.append(nn.Linear(in_features, out_features))
            fc_net_layers.append(nn.ReLU())
            in_features = out_features
        fc_net_layers.append(nn.Linear(in_features, actions_size))
        self.fc_net = nn.Sequential(*fc_net_layers)

    def forward(self, state: torch.Tensor):
        if state.dim() == 1: state = state.view(1, self.input_channels, self.board_height, self.board_width)
        elif state.dim() == 2: state = state.view(-1, self.input_channels, self.board_height, self.board_width)
        elif state.dim() == 3: state = state.unsqueeze(1)
        return self.fc_net(self.conv_net(state))

class PolicyRNN(nn.Module):
    def __init__(self, state_size: int, actions_size: int, hidden_amount: int = 3, layer_size: int = 256, **kwargs):
        super().__init__()
        layers = [("input", nn.Linear(state_size, layer_size)), ("relu0", nn.ReLU())]
        for i in range(hidden_amount):
            layers.append((f"hidden{i}", nn.Linear(layer_size, layer_size)))
            layers.append((f"relu{i+1}", nn.ReLU()))
        layers.append(("output", nn.Linear(layer_size, actions_size)))
        self.model = nn.Sequential(OrderedDict(layers))

    def forward(self, state: torch.Tensor):
        return self.model(state)

# --- Bot Wrapper ---
class PolicyBot:
    def __init__(self, model_path, arch_details):
        self.model_type = arch_details['type']
        if self.model_type == 'cnn':
            self.policy = PolicyCNN(actions_size=7, input_channels=2, **arch_details)
            self.preprocess_func = preprocess_board_cnn
        elif self.model_type == 'rnn':
            self.policy = PolicyRNN(state_size=42, actions_size=7, **arch_details)
            self.preprocess_func = preprocess_board_rnn
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")

        state_dict = torch.load(model_path, map_location=torch.device('cpu'))
        
        # This is a hack to deal with the fact that some models have a value function and others dont
        policy_state_dict = {k: v for k, v in state_dict.items() if 'value_function' not in k}
        # also the rnn model has a "model" key
        policy_state_dict = {k.replace('model.', ''): v for k, v in policy_state_dict.items()}


        self.policy.load_state_dict(policy_state_dict, strict=False)
        self.policy.eval()

    def get_action(self, board, piece):
        ai_input = torch.from_numpy(self.preprocess_func(board, piece)).float().unsqueeze(0)
        with torch.no_grad():
            logits = self.policy.forward(ai_input)[0]
            mask = torch.tensor([0.0 if board[0][c] == 0 else -1e9 for c in range(7)])
            return torch.argmax(logits + mask).item()

# --- Benchmark ---
def generate_random_state(n_moves):
    env = Connect4Env()
    obs, _ = env.reset()
    move_sequence = []
    for _ in range(n_moves):
        if env.done: break
        legal_moves = [c for c in range(env.cols) if env.board[0][c] == 0]
        if not legal_moves: break
        action = random.choice(legal_moves)
        obs, _, _, _, _ = env.step(action)
        move_sequence.append(str(action + 1))
    return obs, env.current_player, "".join(move_sequence)

def run_benchmark(n):
    '''n: Amount of positions to check'''
    save_dir = 'Saves/confusion_matrices'
    os.makedirs(save_dir, exist_ok=True)
    try:
        with open("architectures.json", "r") as f:
            architectures = json.load(f)
    except:
        create_architectures_json()
    with open("architectures.json", "r") as f:
        architectures = json.load(f)

    solved_bot = SolvedBot()
    bots = {"AlphaBetaBot": AlphaBetaBot(depth=4)}

    bot_files = list(architectures.keys())
    bot_files = sorted(bot_files, key=os.path.getmtime, reverse=True)
    bot_files = [f for f in bot_files if 'unused' in f] + bot_files[:50]

    for model_path in tqdm(bot_files, desc="Loading Bots"):
        model_name = os.path.basename(model_path)
        bots[model_name] = PolicyBot(model_path, architectures[model_path])
    
    results = {name: [] for name in bots.keys()}
    ground_truth = []

    N_POSITIONS = n
    test_positions = [generate_random_state(random.randint(5, 20)) for _ in tqdm(range(N_POSITIONS), desc="Generating Positions")]

    for board, player, move_seq in tqdm(test_positions, desc="Running Benchmark"):
        correct_move = solved_bot.get_next_best_move(move_seq)
        if correct_move is None: continue
        ground_truth.append(correct_move - 1)
        for name, bot in bots.items():
            results[name].append(bot.get_action(board.copy(), player))

    for name, bot_moves in results.items():
        if not bot_moves: continue
        min_len = min(len(ground_truth), len(bot_moves))
        acc = accuracy_score(ground_truth[:min_len], bot_moves[:min_len])
        print(f"--- Results for {name} ---\nAccuracy vs SolvedBot: {acc:.2f}")

        cm = confusion_matrix(ground_truth[:min_len], bot_moves[:min_len], labels=list(range(7)))
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(1, 8), yticklabels=range(1, 8))
        plt.title(f'Confusion Matrix for {name}')
        plt.xlabel('Predicted Move'); plt.ylabel('True Move')
        plt.savefig(os.path.join(save_dir, f'cm_{name}.png')); plt.close()

In [ ]:
run_benchmark(1000)

Running Benchmark: 100%|██████████| 100/100 [00:05<00:00, 18.74it/s]


--- Results for AlphaBetaBot ---
Accuracy vs SolvedBot: 0.41
--- Results for Really Good medium CNN.pth ---
Accuracy vs SolvedBot: 0.29
--- Results for best small CNN model.pth ---
Accuracy vs SolvedBot: 0.41
--- Results for Best RNN model.pth ---
Accuracy vs SolvedBot: 0.13
--- Results for connect4_parallel_iter_23000.pth ---
Accuracy vs SolvedBot: 0.35
--- Results for beat_alpha_beta_bot.pth ---
Accuracy vs SolvedBot: 0.37
--- Results for connect4_parallel_iter_22500.pth ---
Accuracy vs SolvedBot: 0.31
--- Results for connect4_parallel_iter_22000.pth ---
Accuracy vs SolvedBot: 0.37
--- Results for connect4_parallel_iter_21500.pth ---
Accuracy vs SolvedBot: 0.34
--- Results for connect4_parallel_iter_21000.pth ---
Accuracy vs SolvedBot: 0.36
--- Results for connect4_parallel_iter_20500.pth ---
Accuracy vs SolvedBot: 0.39
--- Results for connect4_parallel_iter_20000.pth ---
Accuracy vs SolvedBot: 0.39
--- Results for connect4_parallel_iter_2500.pth ---
Accuracy vs SolvedBot: 0.47
--- R